In [1]:
!pip install "setuptools<81.5.0" timm facenet-pytorch grad-cam gradio --only-binary=:all:

INFO: pip is looking at multiple versions of facenet-pytorch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.5 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [facenet-pytorch]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os, glob, random, zipfile
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import timm
from facenet_pytorch import MTCNN
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
print(os.listdir("/kaggle/input"))

['datasets']


In [4]:
import os

# List all dataset subdirectories inside /kaggle/input
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    # Stop after depth 2 so it doesn't flood the output with thousands of video files
    if root.count(os.sep) >= 4:
        dirs.clear()

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/xdxd003
/kaggle/input/datasets/greatgamedota
/kaggle/input/datasets/ahmedelbanby
/kaggle/input/datasets/wish096


In [5]:
BASE = "/kaggle/input/datasets"
for owner in os.listdir(BASE):
    for ds in os.listdir(f"{BASE}/{owner}"):
        p = f"{BASE}/{owner}/{ds}"
        vids = len(glob.glob(f"{p}/**/*.mp4", recursive=True))
        imgs = len(glob.glob(f"{p}/**/*.jpg", recursive=True)) + \
               len(glob.glob(f"{p}/**/*.jpeg", recursive=True)) + \
               len(glob.glob(f"{p}/**/*.png", recursive=True))
        print(f"{owner}/{ds} -> videos: {vids} | images: {imgs}")

xdxd003/ff-c23 -> videos: 7000 | images: 0
greatgamedota/faceforensics -> videos: 0 | images: 20351
ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure -> videos: 6000 | images: 400596
wish096/ff-andcelebdf-frame-dataset-by-wish -> videos: 0 | images: 38940


In [6]:
for owner in os.listdir(BASE):
    for ds in os.listdir(f"{BASE}/{owner}"):
        p = f"{BASE}/{owner}/{ds}"
        print("\n", f"{owner}/{ds}")
        for d in sorted(os.listdir(p))[:10]:
            print("   -", d)


 xdxd003/ff-c23
   - FaceForensics++_C23

 greatgamedota/faceforensics
   - cropped_images

 ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure
   - .upload_cache.json
   - frames
   - videos

 wish096/ff-andcelebdf-frame-dataset-by-wish
   - frame_dataset_v3
   - test_labels.csv
   - train_labels.csv
   - val_labels.csv


In [7]:
A = "/kaggle/input/datasets/ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure"
def peek(path, depth=0, max_depth=3, n=6):
    if depth > max_depth or not os.path.isdir(path):
        return
    for d in sorted(os.listdir(path))[:n]:
        print("  " * depth + "-", d)
        peek(f"{path}/{d}", depth + 1, max_depth, n)

peek(f"{A}/frames", max_depth=2)
print("---- videos ----")
peek(f"{A}/videos", max_depth=2)

- Celeb-DF
  - README.md
  - dataset_info.json
  - manipulated_sequences
    - CelebDF_Synthesis
  - original_sequences
    - youtube
- DFDC_Original
  - dataset_info.json
  - original_sequences
    - youtube
- DFDC_Test
  - dataset_info.json
  - manipulated_sequences
    - DFDC_Unknown
- Method_A
  - dataset_info.json
  - manipulated_sequences
    - Method_A
- Method_B
  - dataset_info.json
  - manipulated_sequences
    - Method_B
- Original_DFDC
  - README.md
  - dataset_info.json
  - original_sequences
    - youtube
---- videos ----
- FaceForensics++
  - dataset_info.json
  - manipulated_sequences
    - Deepfakes
    - Face2Face
    - FaceShifter
    - FaceSwap
    - NeuralTextures
  - original_sequences
    - youtube


In [8]:
G = "/kaggle/input/datasets/greatgamedota/faceforensics"
peek(G, max_depth=3, n=5)

- cropped_images
  - 000_003
    - 0000.png
    - 0025.png
    - 0050.png
    - 0075.png
    - 0100.png
  - 001_870
    - 0000.png
    - 0025.png
    - 0050.png
    - 0075.png
    - 0100.png
  - 002_006
    - 0000.png
    - 0025.png
    - 0050.png
    - 0075.png
    - 0100.png
  - 003_000
    - 0000.png
    - 0025.png
    - 0050.png
    - 0075.png
    - 0100.png
  - 004_982
    - 0000.png
    - 0025.png
    - 0050.png
    - 0075.png
    - 0100.png


In [9]:
W = "/kaggle/input/datasets/wish096/ff-andcelebdf-frame-dataset-by-wish"
df_w = pd.read_csv(f"{W}/train_labels.csv")
print(df_w.shape)
print(df_w.columns.tolist())
print(df_w.head(3))

(32209, 8)
['filepath', 'source', 'label', 'video', 'frame', 'split', 'det_box', 'det_prob']
                                            filepath      source  label  \
0  /kaggle/input/ff-andcelebdf-frame-dataset-by-w...  celeb_fake      1   
1  /kaggle/input/ff-andcelebdf-frame-dataset-by-w...  celeb_fake      1   
2  /kaggle/input/ff-andcelebdf-frame-dataset-by-w...  celeb_fake      1   

            video  frame  split  \
0  id17_id30_0007      0  train   
1  id17_id30_0007    273  train   
2  id45_id40_0000      0  train   

                                             det_box  det_prob  
0  [476.123291015625 77.30122375488281 602.673522...  0.999989  
1  [482.1807861328125 96.54923248291016 601.91821...  0.999685  
2  [328.60443115234375 100.6003189086914 495.7920...  0.999800  


In [10]:
G = "/kaggle/input/datasets/greatgamedota/faceforensics/cropped_images"
vids = os.listdir(G)
fake = [v for v in vids if "_" in v]
real = [v for v in vids if "_" not in v]
print("total:", len(vids), "| with _ :", len(fake), "| without _ :", len(real))
counts = [len(os.listdir(f"{G}/{v}")) for v in vids[:200]]
print("frames per video: min", min(counts), "max", max(counts))

total: 998 | with _ : 998 | without _ : 0
frames per video: min 8 max 56


In [11]:
peek(f"{A}/frames/FaceForensics++", max_depth=3, n=4)
print("---- Celeb-DF ----")
peek(f"{A}/frames/Celeb-DF", max_depth=4, n=3)

---- Celeb-DF ----
- README.md
- dataset_info.json
- manipulated_sequences
  - CelebDF_Synthesis
    - c23
      - frames
        - id0_id16_0000
        - id0_id16_0001
        - id0_id16_0002


In [12]:
CFG = {
    "img_size": 224,
    "num_frames": 8,
    "batch_size": 8,
    "epochs": 10,
    "lr": 1e-4,
    "use_lstm": True,
    "num_real": 300,
    "num_fake": 300,
    "fake_methods": ["Deepfakes", "FaceSwap"],
    "seed": 42,
}

# Video training data (FF++ c23 videos)
VIDEO_ROOT = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"

# Cross-dataset test data (FF++ and Celeb-DF frames with labels)
CELEB_ROOT = "/kaggle/input/datasets/wish096/ff-andcelebdf-frame-dataset-by-wish"

WORK_DIR = "/kaggle/working"
FRAMES_DIR = f"{WORK_DIR}/frames"
os.makedirs(FRAMES_DIR, exist_ok=True)

print(CFG)
print("Frames folder exists:", os.path.exists(FRAMES_DIR))

{'img_size': 224, 'num_frames': 8, 'batch_size': 8, 'epochs': 10, 'lr': 0.0001, 'use_lstm': True, 'num_real': 300, 'num_fake': 300, 'fake_methods': ['Deepfakes', 'FaceSwap'], 'seed': 42}
Frames folder exists: True


In [13]:
for name, path in [("VIDEO_ROOT", VIDEO_ROOT), ("CELEB_ROOT", CELEB_ROOT)]:
    print(name, "exists:", os.path.exists(path))

for d in ["original"] + CFG["fake_methods"]:
    n = len(glob.glob(f"{VIDEO_ROOT}/{d}/*.mp4"))
    print(d, "->", n, "videos")

print(os.listdir(CELEB_ROOT))

VIDEO_ROOT exists: True
CELEB_ROOT exists: True
original -> 1000 videos
Deepfakes -> 1000 videos
FaceSwap -> 1000 videos
['frame_dataset_v3', 'test_labels.csv', 'train_labels.csv', 'val_labels.csv']


In [14]:
X = VIDEO_ROOT
methods = CFG["fake_methods"]

partner = {}
for f in os.listdir(f"{X}/Deepfakes"):
    a, b = f[:-4].split("_")
    partner[a], partner[b] = b, a

groups = sorted({tuple(sorted((a, b))) for a, b in partner.items()})
random.Random(42).shuffle(groups)

split_of = {}
n = len(groups)
for i, g in enumerate(groups):
    s = "train" if i < 0.72 * n else ("val" if i < 0.86 * n else "test")
    for v in g:
        split_of[v] = s

rows = []
for f in os.listdir(f"{X}/original"):
    vid = f[:-4]
    if vid in split_of:
        rows.append((f"{X}/original/{f}", 0, "real", vid, split_of[vid]))
for m in methods:
    for f in os.listdir(f"{X}/{m}"):
        a = f[:-4].split("_")[0]
        if a in split_of:
            rows.append((f"{X}/{m}/{f}", 1, m, f[:-4], split_of[a]))

df = pd.DataFrame(rows, columns=["path", "label", "method", "vid", "split"])

real_q = {"train": 210, "val": 45, "test": 45}
fake_q = {"train": 105, "val": 23, "test": 23}

parts = []
for s, q in real_q.items():
    parts.append(df[(df.label == 0) & (df.split == s)].sample(q, random_state=42))
for s, q in fake_q.items():
    for m in methods:
        parts.append(df[(df.method == m) & (df.split == s)].sample(q, random_state=42))

small = pd.concat(parts).reset_index(drop=True)
small.to_csv(f"{WORK_DIR}/split.csv", index=False)
print(small.groupby(["split", "label"]).size())

split  label
test   0         45
       1         46
train  0        210
       1        210
val    0         45
       1         46
dtype: int64


In [15]:
from tqdm.auto import tqdm

NPY_DIR = f"{WORK_DIR}/npy"
os.makedirs(NPY_DIR, exist_ok=True)
mtcnn = MTCNN(keep_all=False, device=DEVICE)

def crop_face(frame_rgb, margin=0.25):
    boxes, _ = mtcnn.detect(frame_rgb)
    if boxes is None:
        return None
    x1, y1, x2, y2 = boxes[0]
    w, h = x2 - x1, y2 - y1
    H, W = frame_rgb.shape[:2]
    x1, y1 = int(max(0, x1 - margin * w)), int(max(0, y1 - margin * h))
    x2, y2 = int(min(W, x2 + margin * w)), int(min(H, y2 + margin * h))
    face = frame_rgb[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, (CFG["img_size"], CFG["img_size"]))

def video_to_npy(path, n=CFG["num_frames"]):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n:
        cap.release()
        return None
    faces = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        face = crop_face(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) if ok else None
        if face is None and faces:
            face = faces[-1]
        if face is not None:
            faces.append(face)
    cap.release()
    if len(faces) < n:
        return None
    return np.stack(faces).astype(np.uint8)

LIMIT = 20   # test with 20 first, then set to None for all videos
data = small if LIMIT is None else small.sample(LIMIT, random_state=1)

npy_paths = {}
for _, r in tqdm(data.iterrows(), total=len(data)):
    out = f"{NPY_DIR}/{r['method']}_{r['vid']}.npy"
    if not os.path.exists(out):
        arr = video_to_npy(r["path"])
        if arr is None:
            continue
        np.save(out, arr)
    npy_paths[r["path"]] = out

clean = small[small["path"].isin(npy_paths)].copy()
clean["npy_path"] = clean["path"].map(npy_paths)
clean.to_csv(f"{WORK_DIR}/split_clean.csv", index=False)
print("kept:", len(clean), "of", len(data))
print(clean.groupby(["split", "label"]).size())

  0%|          | 0/20 [00:00<?, ?it/s]

kept: 20 of 20
split  label
test   0        1
       1        2
train  0        7
       1        7
val    0        1
       1        2
dtype: int64


In [16]:
from tqdm.auto import tqdm

NPY_DIR = f"{WORK_DIR}/npy"
os.makedirs(NPY_DIR, exist_ok=True)
mtcnn = MTCNN(keep_all=False, device=DEVICE)

def crop_face(frame_rgb, margin=0.25):
    boxes, _ = mtcnn.detect(frame_rgb)
    if boxes is None:
        return None
    x1, y1, x2, y2 = boxes[0]
    w, h = x2 - x1, y2 - y1
    H, W = frame_rgb.shape[:2]
    x1, y1 = int(max(0, x1 - margin * w)), int(max(0, y1 - margin * h))
    x2, y2 = int(min(W, x2 + margin * w)), int(min(H, y2 + margin * h))
    face = frame_rgb[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, (CFG["img_size"], CFG["img_size"]))

def video_to_npy(path, n=CFG["num_frames"]):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < n:
        cap.release()
        return None
    faces = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        face = crop_face(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) if ok else None
        if face is None and faces:
            face = faces[-1]
        if face is not None:
            faces.append(face)
    cap.release()
    if len(faces) < n:
        return None
    return np.stack(faces).astype(np.uint8)

LIMIT = None   # test with 20 first, then set to None for all videos
data = small if LIMIT is None else small.sample(LIMIT, random_state=1)

npy_paths = {}
for _, r in tqdm(data.iterrows(), total=len(data)):
    out = f"{NPY_DIR}/{r['method']}_{r['vid']}.npy"
    if not os.path.exists(out):
        arr = video_to_npy(r["path"])
        if arr is None:
            continue
        np.save(out, arr)
    npy_paths[r["path"]] = out

clean = small[small["path"].isin(npy_paths)].copy()
clean["npy_path"] = clean["path"].map(npy_paths)
clean.to_csv(f"{WORK_DIR}/split_clean.csv", index=False)
print("kept:", len(clean), "of", len(data))
print(clean.groupby(["split", "label"]).size())

  0%|          | 0/602 [00:00<?, ?it/s]

kept: 602 of 602
split  label
test   0         45
       1         46
train  0        210
       1        210
val    0         45
       1         46
dtype: int64


In [17]:
X = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
for d in sorted(os.listdir(X)):
    p = f"{X}/{d}"
    if os.path.isdir(p):
        files = os.listdir(p)
        print(d, "->", len(files), "items | e.g.", files[:3])
    else:
        print(d, "(file)")

DeepFakeDetection -> 1000 items | e.g. ['01_02__meeting_serious__YVGY8LOK.mp4', '01_02__outside_talking_still_laughing__YVGY8LOK.mp4', '01_02__talking_against_wall__YVGY8LOK.mp4']
Deepfakes -> 1000 items | e.g. ['000_003.mp4', '001_870.mp4', '002_006.mp4']
Face2Face -> 1000 items | e.g. ['000_003.mp4', '001_870.mp4', '002_006.mp4']
FaceShifter -> 1000 items | e.g. ['000_003.mp4', '001_870.mp4', '002_006.mp4']
FaceSwap -> 1000 items | e.g. ['000_003.mp4', '001_870.mp4', '002_006.mp4']
NeuralTextures -> 1000 items | e.g. ['000_003.mp4', '001_870.mp4', '002_006.mp4']
csv -> 10 items | e.g. ['DeepFakeDetection.csv', 'Deepfakes.csv', 'FF++_Metadata.csv']
original -> 1000 items | e.g. ['000.mp4', '001.mp4', '002.mp4']


In [18]:
csvs = glob.glob(f"{X}/**/*.csv", recursive=True)
print(csvs)
c = pd.read_csv(csvs[0])
print(c.shape)
print(c.head(3))

['/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/DeepFakeDetection.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/Deepfakes.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/FF++_Metadata.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/FF++_Metadata_Shuffled.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/Face2Face.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/FaceShifter.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/FaceSwap.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/Mean_Data.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/NeuralTextures.csv', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv/original.csv']
(1000, 8)
   Unnamed: 0                                          File Path Label  \
0           0  DeepFakeDetection/01_02__meeting_serious__YVGY...  FAKE   
1           1  DeepFa